# Heat Notebook
12 easy to following steps and repeatable with any city or area

## 1. Setup: Imports / Initializing

In [1]:
import geemap
import ee
import geopandas as gpd

/Users/tinluong/Desktop/dfwHeat/DFW_UrbanHeat/venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
#Uncomment and run Authenticate once then you don't need it anymore
#ee.Authenticate()
ee.Initialize(project = "dfw-urbanheat")

## 2. Parameters
ONLY change this section

In [3]:
#STUDY AREA
STATE_FIPS = '48'                   #State
COUNTY_FIPS = ['113', '439']        #County codes
MAP_CENTER = (32.8, -96.9)          #Middle of DFW
MAP_ZOOM = 10                       #Zoom

#TIME PERIOD
START_DATE = "2020-01-01"           #Date Start
END_DATE = "2024-12-31"             #Date End
START_MONTH = 6                     #Month Start
END_MONTH = 8                       #Month End
#--------------------Don't--Change-----------------------------------
LANDSAT8 = "LANDSAT/LC08/C02/T1_L2" #Landsat 8 (8 & 9 are combined)
LANDSAT9 = "LANDSAT/LC09/C02/T1_L2" #Landsat 9 (8 & 9 are combined)
#--------------------------------------------------------------------

#IMAGE FILTERING
MAX_CLOUD_COVER = 20                #Max cloud cover allowed (%)

#PROCESSING 
SCALE = 30                          #Landsat resolution

#Temperature layer visual
LSAT_VIS = {"min": 99, "max": 129, "palette": ["#4575b4", "#91bfdb", "#e0f3f8", "#ffffbf", "#fee090", "#fc8d59", "#d73027"]}
#NDVI layer visual
NDVI_VIS = {'min': 0.1, 'max': .8, "palette": ['#edf8fb', '#ccece6', '#99d8c9', '#66c2a4', '#41ae76', '#238b45', '#005824']}



## 3. Study Area: Ex. (DFW)

In [5]:
Map = geemap.Map(center=(MAP_CENTER), zoom=MAP_ZOOM)
STUDY_AREA = (
    ee.FeatureCollection("TIGER/2020/TRACT")
    .filter(ee.Filter.eq("STATEFP", STATE_FIPS))
    .filter(ee.Filter.inList("COUNTYFP", COUNTY_FIPS))
    )
#DFW_area.size().getInfo()


## 4. Loading Landsat Data
What's loaded in: Landsat 8 and 9 collection 2 and level 2 scenes merged and filtered with cloud cover and start end dates. (able to change)

In [6]:
landsat_8 = ee.ImageCollection(LANDSAT8) #landsat 8
landsat_9 = ee.ImageCollection(LANDSAT9) #landsat 9

land_combined = (
    landsat_8.merge(landsat_9)
    .filterBounds(STUDY_AREA.geometry().bounds())
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.calendarRange(START_MONTH, END_MONTH, 'month'))
    .filter(ee.Filter.lt("CLOUD_COVER", MAX_CLOUD_COVER))
    )
#land_combined.size().getInfo() #amount of usable images you get taking in the cloud cover & summer months


## 5. Cloud & Water Masking and Unit Conversion

In [7]:
def cloud_water_masking(image): #adding water too
    qa = image.select("QA_PIXEL")
    cloud_edge = 1 << 1
    cloud_thin = 1 << 2
    cloud = 1 << 3
    cloud_shadow = 1 << 4
    water = 1 << 7

    mask = (
        qa.bitwiseAnd(cloud_edge).eq(0)
        .And(qa.bitwiseAnd(cloud_thin).eq(0))
        .And(qa.bitwiseAnd(cloud).eq(0))
        .And(qa.bitwiseAnd(cloud_shadow).eq(0))
        .And(qa.bitwiseAnd(water).eq(0))
    )
    return image.updateMask(mask)

landsat_masked = land_combined.map(cloud_water_masking)



## 6. Summer Surface Temperature
Notably higher temperatures than air temperature

In [8]:
def lsat_convert(image):
    #converting temperature into F
    temp = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).multiply(1.8).add(32).rename('LST_F')
    temp_f = image.addBands(temp)
    return temp_f

landsat_temp = landsat_masked.map(lsat_convert)

landsat_median = landsat_temp.median() #this landsat for doing different versions like the ndvi
landsat_median_temp = landsat_median.select('LST_F').clip(STUDY_AREA)

In [9]:
landsat_stats = landsat_median_temp.reduceRegion(
    scale = SCALE,
    reducer= ee.Reducer.percentile([5,95]),
    maxPixels = 1e9,
    geometry = STUDY_AREA.geometry()
)
print(landsat_stats.getInfo()) #shows the percentile & degrees in F
#5th percentile is around 102 F and 95th percentile is around 126 F

{'LST_F_p5': 101.25069774751542, 'LST_F_p95': 125.24183192710909}


In [10]:
Map.add_layer(landsat_median_temp, LSAT_VIS, "Summer median LST (°F)")
Map.add_colorbar(LSAT_VIS, label="Land surface temperature (°F)")
#Map

## 7. NDVI
SR_B5: Near-Infrared (NIR) | SR_B4: Red

In [11]:
#Calculates the NDVI from summer's median surface temperature
def ndvi_convert(image):
    band_nir_red = image.select(['SR_B5', 'SR_B4']).multiply(0.0000275).add(-0.2)
    ndvi = band_nir_red.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI')
    return ndvi

landsat_median_ndvi = ndvi_convert(landsat_median).clip(STUDY_AREA)

In [12]:
#statistics of NDVI on the 5th and the 95th percentile
landsat_ndvi_stats = landsat_median_ndvi.reduceRegion(
    scale = SCALE,
    reducer = ee.Reducer.percentile([5,95]),
    maxPixels = 1e9,
    geometry = STUDY_AREA.geometry()
)
landsat_ndvi_stats.getInfo() 

{'NDVI_p5': 0.11331122458980061, 'NDVI_p95': 0.785136648329978}

In [13]:
#VISUAL
Map.add_layer(landsat_median_ndvi, NDVI_VIS, "NDVI")
Map.add_colorbar(NDVI_VIS, label='NDVI')

## 8. Tract-level Statistics

In [14]:
landsat_temp_ndvi = landsat_median_ndvi.addBands(landsat_median_temp)
#landsat_temp_ndvi.bandNames().getInfo()

temp_ndvi_stats = landsat_temp_ndvi.reduceRegions(
    collection = STUDY_AREA,
    reducer = ee.Reducer.mean().combine(reducer2= ee.Reducer.count(), sharedInputs= True),
    scale = SCALE,
)

#temp_ndvi_stats.first().getInfo()


In [15]:
tract_stats = temp_ndvi_stats.select(['GEOID', 'COUNTYFP', 'NDVI_count', 'NDVI_mean', 'ALAND', 'LST_F_mean', 'AWATER'])
tract_ndvi_temp = geemap.ee_to_gdf(tract_stats)
tract_ndvi_temp.to_file('data/processed/tract_stats.geojson', driver = 'GeoJSON' )


In [16]:
ndvi_sorted = tract_ndvi_temp.sort_values('NDVI_count')
ndvi_sorted.describe()

,ALAND,AWATER,LST_F_mean,NDVI_count,NDVI_mean
count,1.094000e+03,1.094000e+03,1094.000000,1094.000000,1094.000000
mean,4.115428e+06,1.752125e+05,117.402212,5703.327239,0.422342
std,7.025292e+06,8.957204e+05,3.687174,9638.772839,0.087986
min,1.449260e+05,0.000000e+00,100.704591,230.000000,0.158035
25%,1.542734e+06,0.000000e+00,115.303137,2159.750000,0.364078
50%,2.554630e+06,0.000000e+00,117.817024,3533.500000,0.431441
75%,3.998992e+06,1.365375e+04,119.982904,5561.000000,0.485330
max,1.337815e+08,1.096252e+07,126.668202,180373.000000,0.713988


## 9. Joining with Census Data

In [17]:
gdf = gpd.read_file('data/processed/tract_stats.geojson')
gdf.head()

,ALAND,AWATER,COUNTYFP,GEOID,LST_F_mean,NDVI_count,NDVI_mean,geometry
0,3077392,511268,113,48113000100,109.661783,4435,0.576505,"POLYGON ((-96.75301 32.81556, -96.75299 32.815..."
1,1651595,2154,113,48113002200,121.341526,2331,0.265247,"POLYGON ((-96.79236 32.78813, -96.79235 32.787..."
2,2107023,1046,113,48113004400,111.606177,2953,0.559991,"POLYGON ((-96.85681 32.75569, -96.85664 32.755..."
3,1972793,11228,113,48113004500,116.059668,2774,0.474897,"POLYGON ((-96.87422 32.73447, -96.87422 32.734..."
4,712964,143,113,48113004600,115.917586,994,0.492765,"POLYGON ((-96.85145 32.7466, -96.85122 32.7465..."


## 10. Analysis & Results

## 11. Visualization

## 12. Exporting